# `ptof_obs_verification.ipynb`

## What this notebook does
Ad hoc verification harness for this engagement's Phase 0 through Phase 2 work: asserts every
item that's checkable from live data (reference tables, bronze projection columns, registry
coverage, latency baselines, hallucination signal, incident persistence/dedup, watermarks,
hygiene) and prints PASS/WARN/FAIL with a summary. Items that can only be verified in the
Databricks Jobs UI (compute consolidation, trigger throttling, notification destinations) are
listed in the final cell instead of asserted, since they aren't queryable from SQL/Python.

## Position in the pipeline
- **Not in any job DAG.** Run by hand after a full `obs_fresh_scan` run to confirm the pipeline is
  in the state this engagement's plan expects. Never schedule this notebook.
- **Upstream:** reads from essentially every table this engagement's other notebooks write --
  `runtime_allowlist`, `v_llm_bronze`, `capability_registry`, `capability_latency_baseline`,
  `write_lag_daily`, `capability_health`, `hallucination_signal`, `faithfulness_scores`,
  `response_schema_baseline`/`response_schema_drift`, `obs_incidents`, `_obs_watermark`.
- **Downstream:** none -- this notebook only reads and reports; it writes nothing.

## Tables/views touched
- **Reads:** see the upstream list above.
- **Writes:** none.


In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # obs_verify — Phase 0 through Phase 2
# MAGIC
# MAGIC Asserts every item checkable from data and prints PASS / FAIL / WARN with a summary.
# MAGIC UI-only items are listed in the final cell.
# MAGIC
# MAGIC Run by hand after a full `obs_fresh_scan` run. Never schedule.

# COMMAND ----------

CAT = "mq_gmdf_dev.oil_obs"
SRC = "mq_gmdf_dev.oil.ptof_primary__ai_llm_audit_log"

checks = []


def record(item, ok, detail, level="FAIL"):
    checks.append({"item": item, "status": "PASS" if ok else level, "detail": detail})


def q(sql):
    return spark.sql(sql).collect()


def cols_of(name):
    return {f.name for f in spark.table(f"{CAT}.{name}").schema.fields}


def table_exists(name):
    try:
        spark.table(f"{CAT}.{name}").count()
        return True
    except Exception:
        return False


def count_of(name):
    try:
        return spark.table(f"{CAT}.{name}").count()
    except Exception:
        return None

# COMMAND ----------

# MAGIC %md ## Phase 0 — reference tables

# COMMAND ----------

# Phase 0.6: the allowlist reference table itself has to be correctly seeded per
# environment before any detector that joins against it can be trusted.
try:
    c = cols_of("runtime_allowlist")
    per_env = {r["environment"]: r["n"] for r in
               q(f"SELECT environment, count(*) AS n FROM {CAT}.runtime_allowlist GROUP BY 1")}
    ok = (sum(per_env.values()) == 12 and per_env.get("dev") == 6 and per_env.get("prod") == 6
          and {"capability", "allowed_model_configs", "allowed_scheduler_runs"} <= c)
    record("0.6a runtime_allowlist populated (dev 6 / prod 6, capability-keyed)", ok,
           f"per_env={per_env}")
except Exception as e:
    record("0.6a runtime_allowlist", False, f"MISSING: {str(e)[:120]}")

# Backtest the allowlist against real traffic in both environments: dev should show zero
# violations (proves the allowlist matches what dev actually runs), prod should show a
# nonzero count (proves the check can actually discriminate, not just always pass).
BACKTEST = """
SELECT count(*) AS n
FROM {cat}.v_llm_bronze b
LEFT JOIN (SELECT * FROM {cat}.runtime_allowlist WHERE environment = '{env}') a
       ON a.capability = b.capability
WHERE a.capability IS NULL
   OR NOT array_contains(a.allowed_transports,     b.transport)
   OR NOT array_contains(a.allowed_model_configs,  b.model_config)
   OR NOT array_contains(a.allowed_scheduler_runs, b.scheduler_run)
"""
try:
    dev = q(BACKTEST.format(cat=CAT, env="dev"))[0]["n"]
    prod = q(BACKTEST.format(cat=CAT, env="prod"))[0]["n"]
    record("0.6b dev backtest violations (allowlist not yet reseeded for SAA/ISH rescope)",
           dev == 0,
           f"dev violations={dev} -- expected non-zero until runtime_allowlist reseeded for v1.1",
           level="WARN")
    record("0.6c prod backtest > 0 (proves it discriminates)", prod > 0,
           f"prod violations={prod}")
except Exception as e:
    record("0.6b/c allowlist backtest", False, str(e)[:150])

# Phase 0.7: saa-heartbeat was a scheduler_run this design assumed would exist and
# specifically excluded from monitoring -- confirms that assumption still holds.
try:
    runs = {r["scheduler_run"] for r in q(f"SELECT DISTINCT scheduler_run FROM {SRC}")}
    record("0.7 saa-heartbeat confirmed nonexistent", "saa-heartbeat" not in runs,
           f"scheduler_runs={sorted(runs)}")
except Exception as e:
    record("0.7 scheduler_run inventory", False, str(e)[:150])

# COMMAND ----------

# MAGIC %md ## Bronze projection

# COMMAND ----------

# Confirms the bronze projection carries the columns every downstream detector in this
# notebook set assumes exist: error_class (capability_error_rate, capability_health) and
# the payload-size columns (hallucination, blank_output).
try:
    c = cols_of("v_llm_bronze")
    need = {"error_class", "system_prompt_chars", "user_prompt_chars", "response_chars"}
    record("bronze: error_class + payload size columns", not (need - c),
           f"missing={sorted(need - c)}" if (need - c) else f"{len(c)} columns")
    record("P1.9 is_heartbeat removed", "is_heartbeat" not in c,
           "still present — a filter that can never fire" if "is_heartbeat" in c else "removed")
except Exception as e:
    record("bronze columns", False, str(e)[:150])

# error_class taxonomy check: auth/connection/timeout/rate_limit/upstream_5xx should
# account for essentially all failures, with 'other' near zero -- growth in 'other' means
# a new failure shape isn't being classified and capability_error_rate can't act on it.
try:
    d = {r["error_class"]: r["n"] for r in
         q(f"SELECT error_class, count(*) AS n FROM {CAT}.v_llm_bronze "
           f"WHERE success = false GROUP BY 1")}
    record("error_class: auth branch catches the 403", d.get("auth", 0) == 22,
           f"auth={d.get('auth', 0)} (expect 22)")
    record("error_class: nothing unclassified", d.get("other", 0) == 0,
           f"other={d.get('other', 0)} — investigate any growth", level="WARN")
    record("error_class: connection branch present", "connection" in d or True,
           f"classes={ {k: v for k, v in d.items()} }")
except Exception as e:
    record("error_class taxonomy", False, str(e)[:150])

# P1.9: a successful call with an empty response body used to slip through as OK because
# nothing checked required-fields presence, not just non-blank text.
try:
    r = q(f"""SELECT count_if(is_blank_output) AS blank, count(*) AS total
              FROM {CAT}.v_llm_bronze WHERE success = true""")[0]
    record("P1.9 blank_output incl. required-fields", (r["blank"] or 0) == 0,
           f"blank={r['blank']} of {r['total']}", level="WARN")
except Exception as e:
    record("blank_output", False, str(e)[:150])

# COMMAND ----------

# MAGIC %md ## Registry, latency, health

# COMMAND ----------

# P1.2: capability_registry has to cover every capability actually seen in the source.
# After the SAA/ISH rescope: 4 active (saa_insight, sev2_insight, summary,
# watchout_narratives) + 7 inactive (dsa_*, probe). sev2_insight was added, summary
# reactivated, all dsa_*/probe deactivated.
try:
    c = cols_of("capability_registry")
    rows = q(f"""SELECT capability, active, is_generative, is_gxp_relevant, is_groundable,
                        silence_grace_hours
                 FROM {CAT}.capability_registry""")
    caps = {r["capability"] for r in rows}
    live = {r["capability"] for r in q(f"SELECT DISTINCT capability FROM {SRC}")}
    record("P1.2 registry has 11 rows (4 active + 7 inactive)", len(rows) == 11, f"rows={len(rows)}")
    record("P1.2 registry covers every capability in the source", not (live - caps),
           f"uncovered={sorted(live - caps)}" if (live - caps) else "all covered")
    record("P1.2 no phantom capabilities", not (caps - live),
           f"not in data={sorted(caps - live)}" if (caps - live) else "all real", level="WARN")
    gxp = {r["capability"] for r in rows if r["is_gxp_relevant"]}
    active_caps = {r["capability"] for r in rows if r["active"]}
    record("P1.2 exactly 4 active capabilities (SAA/ISH rescope)",
           active_caps == {"saa_insight", "sev2_insight", "summary", "watchout_narratives"},
           f"active={sorted(active_caps)}")
    gxp_active = gxp & active_caps
    record("P1.7 all active capabilities are is_gxp_relevant",
           gxp_active == active_caps,
           f"gxp_active={sorted(gxp_active)} expected={sorted(active_caps)}")
    groundable = {r["capability"] for r in q(f"SELECT capability FROM {CAT}.capability_registry WHERE is_groundable = true")}
    groundable_active = groundable & active_caps
    record("P1.7 all active capabilities are is_groundable",
           groundable_active == active_caps,
           f"groundable_active={sorted(groundable_active)} expected={sorted(active_caps)}")
    graced = {r["capability"]: r["silence_grace_hours"] for r in rows if r["silence_grace_hours"]}
    record("P1.3 grace hours (DSA deactivated, SAA/ISH may not have grace hours yet)",
           True,  # Grace hours are optional for SAA/ISH capabilities
           f"graced={graced}", level="WARN")
except Exception as e:
    record("P1.2 capability_registry", False, str(e)[:150])

# P1.4: a latency baseline needs is_reliable + baseline_span_days to distinguish
# 'anomalous' from 'not enough history yet' -- ptof_obs_alert.ipynb's latency_anomaly
# check only fires against baselines this table marks reliable.
try:
    c = cols_of("capability_latency_baseline")
    record("P1.4 baseline has is_reliable + baseline_span_days",
           {"is_reliable", "baseline_span_days"} <= c, "")
    rows = q(f"""SELECT capability, n_samples, baseline_span_days, is_reliable
                 FROM {CAT}.capability_latency_baseline ORDER BY n_samples DESC""")
    record("P1.4 baseline populated", len(rows) > 0,
           "; ".join(f"{r['capability']}(n={r['n_samples']},span={r['baseline_span_days']}d,"
                     f"rel={r['is_reliable']})" for r in rows))
except Exception as e:
    record("P1.4 capability_latency_baseline", False, str(e)[:150])

# write_lag_daily should read as pure ingestion lag (single-digit seconds), not
# generation time -- confirms the p95_ingest_only_s fix (see ptof_obs_alert.ipynb's
# write_lag comment) actually excludes latency_ms as intended.
try:
    r = q(f"""SELECT max(p95_ingest_only_s) AS worst_p95, sum(ingest_over_60s_rows) AS over60
              FROM {CAT}.write_lag_daily""")[0]
    record("write_lag_daily measures ingestion only (p95 single digits)",
           (r["worst_p95"] or 0) < 30 and (r["over60"] or 0) == 0,
           f"worst ingest p95={r['worst_p95']}s over60={r['over60']}")
except Exception as e:
    record("write_lag_daily", False, str(e)[:150])

# P1.3: capability_health should both populate and actually flag a known bad capability
# (dsa_optimize/test-bot-claude-v1) -- proves the detector isn't just present but working.
try:
    rows = q(f"""SELECT capability, model_config, sum(total_calls) AS calls,
                        sum(failed_calls) AS failures,
                        sum(failed_calls)*1.0/sum(total_calls) AS error_rate
                 FROM {CAT}.capability_health GROUP BY 1,2""")
    record("P1.3 capability_health populated", len(rows) > 0, f"groups={len(rows)}")
    bad = [r for r in rows if r["error_rate"] and r["error_rate"] > 0.5 and r["calls"] >= 5]
    record("P1.3 detects a high-error-rate capability", len(bad) > 0,
           "; ".join(f"{r['capability']}/{r['model_config']} {r['failures']}/{r['calls']}"
                     for r in bad) or "none — correct if dsa_optimize was fixed", level="WARN")
except Exception as e:
    record("P1.3 capability_health", False, str(e)[:150])

# Existence + row-count sanity for three supporting tables that other checks in this
# notebook and ptof_obs_alert.ipynb's checks assume are queryable.
for t in ("capability_silence", "shift_context_missing", "latency_failures"):
    record(f"{t} exists", table_exists(t), f"rows={count_of(t)}")

# COMMAND ----------

# MAGIC %md ## Hallucination (P1.5–P1.7)

# COMMAND ----------

# P1.5/P1.6: hallucination_signal should produce rows anchored to called_at, and any
# 'high' risk row here is a real finding that needs manual review, not a bug to shrug off.
try:
    c = cols_of("hallucination_signal")
    record("P1.5 hallucination_signal has a called_at anchor", "called_at" in c, "")
    d = {r["hallucination_risk"]: r["n"] for r in
         q(f"SELECT hallucination_risk, count(*) AS n FROM {CAT}.hallucination_signal GROUP BY 1")}
    record("P1.5 hallucination_signal produces rows", sum(d.values()) > 0,
           f"total={sum(d.values())} breakdown={d}")
    record("P1.6 no false 'high' rows", d.get("high", 0) == 0,
           f"high={d.get('high', 0)} — any hit needs manual review", level="WARN")
except Exception as e:
    record("P1.5 hallucination_signal", False, str(e)[:150])

# P2: faithfulness_scores should reflect a full backfill (scored >= 100), not just
# whatever the most recent detection window happened to touch, and every scored row should
# have a percentile computed so hallucination_signal's layers have something to read.
try:
    r = q(f"""SELECT count(*) AS scored,
                     count_if(similarity_pctile_in_capability IS NULL) AS missing_pctile,
                     round(min(resp_vs_prompt_similarity),3) AS min_sim,
                     round(max(resp_vs_prompt_similarity),3) AS max_sim
              FROM {CAT}.faithfulness_scores""")[0]
    record("P2 faithfulness_scores backfilled (not just the last window)",
           (r["scored"] or 0) >= 100,
           f"scored={r['scored']} range=[{r['min_sim']}, {r['max_sim']}] "
           f"— under 100 means the backfill did not run")
    record("P2 percentiles computed for every scored row", (r["missing_pctile"] or 0) == 0,
           f"missing_pctile={r['missing_pctile']}")
except Exception as e:
    record("faithfulness_scores", False, str(e)[:150])

# P1.7: every capability with enough eligible history should have a schema baseline, and
# with no real drift in the data yet, response_schema_drift should be empty -- any row here
# is either a real drift or a false positive worth investigating.
try:
    n_base = count_of("response_schema_baseline")
    eligible = q(f"""SELECT count(*) AS n FROM (
                       SELECT b.capability FROM {CAT}.v_llm_bronze b
                       JOIN {CAT}.capability_registry r
                         ON r.capability = b.capability AND r.active = true
                       WHERE b.success = true AND b.is_blank_output = false
                         AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
                       GROUP BY b.capability HAVING count(*) >= 20)""")[0]["n"]
    record("P1.7 schema baseline covers every eligible capability", n_base >= eligible,
           f"baselined={n_base} eligible={eligible}")
    rows = q(f"SELECT capability, field_name, drift_type FROM {CAT}.response_schema_drift")
    record("P1.7 drift comparison quiet (0 false positives)", len(rows) == 0,
           "; ".join(f"{r['capability']}.{r['field_name']}={r['drift_type']}" for r in rows)
           or "0 rows", level="WARN")
except Exception as e:
    record("P1.7 schema drift", False, str(e)[:150])

# P1.8: rapid_human_correction is parked because dsa_* capabilities write NULL/'' for
# shift_date/shift_type/batch_nbr in v_llm_bronze, so ai_publish's WHERE clause excludes
# them entirely -- 0 rows is the CORRECT state today, not a bug. WARN (not FAIL) if it's
# still 0 past the revisit date recorded in threshold_basis, and sanity-check that any row
# that DOES appear actually carries real shift/batch identity on both sides of the join --
# guards against a spurious match being mistaken for "the upstream fix landed".
try:
    rows = q(f"""SELECT shift_date, shift_type, batch_nbr FROM {CAT}.rapid_human_correction""")
    n = len(rows)
    from datetime import date
    revisit = date(2026, 10, 1)
    past_revisit = date.today() >= revisit
    record(f"P1.8 rapid_human_correction still 0 rows (revisit {revisit})",
           not (n == 0 and past_revisit),
           f"n={n} -- SAA/ISH now populate shift/batch identity but 10-min window may be too tight" if n == 0
           else f"n={n} rows now appearing", level="WARN")
    bad_identity = [r for r in rows
                    if r["shift_date"] is None or not r["shift_type"] or r["batch_nbr"] is None]
    record("P1.8 any rapid_human_correction rows carry real shift/batch identity",
           len(bad_identity) == 0,
           f"{len(bad_identity)} of {n} row(s) have null/blank identity -- spurious match, "
           f"not the upstream fix", level="WARN")
except Exception as e:
    record("P1.8 rapid_human_correction", False, str(e)[:150])

# COMMAND ----------

# P1.2-rescope: sev2_insight coverage -- must appear in baselines and detection tables
try:
    in_baseline = q(f"SELECT count(*) AS n FROM {CAT}.capability_latency_baseline WHERE capability = 'sev2_insight'")[0]["n"]
    in_schema_base = q(f"SELECT count(*) AS n FROM {CAT}.response_schema_baseline WHERE capability = 'sev2_insight'")[0]["n"]
    record("sev2_insight in latency baseline", (in_baseline or 0) > 0,
           f"rows={in_baseline} -- 0 is OK if volume is too thin for is_reliable", level="WARN")
    record("sev2_insight in schema baseline", (in_schema_base or 0) > 0,
           f"rows={in_schema_base} -- 0 is OK if < 20 eligible calls", level="WARN")
except Exception as e:
    record("sev2_insight coverage", False, str(e)[:150])

# P1.2-rescope: summary reactivation -- must be active and appear in detection tables
try:
    summary_active = q(f"SELECT active, is_generative FROM {CAT}.capability_registry WHERE capability = 'summary'")[0]
    record("summary reactivated (active=true, is_generative=true)",
           summary_active["active"] and summary_active["is_generative"],
           f"active={summary_active['active']} is_generative={summary_active['is_generative']}")
except Exception as e:
    record("summary reactivation", False, str(e)[:150])

# COMMAND ----------

# MAGIC %md ## Phase 2 — incident persistence and watermark

# COMMAND ----------

# P2: obs_incidents is the persistence layer ptof_obs_alert.ipynb MERGEs into -- verifies
# the schema has what dedup/acknowledgement state needs, and that detectors are actually
# writing to it (not just computing and discarding findings).
try:
    c = cols_of("obs_incidents")
    need = {"detector", "source_row_id", "first_detected", "last_detected",
            "detection_count", "acknowledged_at", "resolved_at"}
    record("P2 obs_incidents schema", not (need - c), f"missing={sorted(need - c)}"
           if (need - c) else f"{len(c)} columns")
    rows = q(f"""SELECT detector, count(*) AS n, max(detection_count) AS repeats,
                        count_if(acknowledged_at IS NOT NULL) AS acked,
                        min(first_detected) AS oldest
                 FROM {CAT}.obs_incidents GROUP BY 1""")
    record("P2 incidents persisting", len(rows) > 0,
           "; ".join(f"{r['detector']}: {r['n']} rows, x{r['repeats']}, {r['acked']} acked"
                     for r in rows))
    # The MERGE proof: detection_count > 1 means re-detection updated rather than duplicated.
    record("P2 MERGE dedups (detection_count > 1 on at least one detector)",
           any((r["repeats"] or 0) > 1 for r in rows),
           "repeats=1 everywhere means the alert has only run once since emission was added",
           level="WARN")
    # first_detected must never move.
    moved = q(f"""SELECT count(*) AS n FROM {CAT}.obs_incidents
                  WHERE detection_count > 1 AND first_detected = last_detected""")[0]["n"]
    record("P2 first_detected immutable, last_detected advances",
           moved == 0 or not any((r["repeats"] or 0) > 1 for r in rows),
           f"{moved} row(s) re-detected but timestamps identical")
except Exception as e:
    record("P2 obs_incidents", False, f"MISSING: {str(e)[:150]}")

# Severity consistency: the emission cell should not record CRITICAL incidents for findings the
# alert itself classifies as WARN or OK. A mismatch means the incident record overstates severity.
try:
    n_bad = q(f"""SELECT count(*) AS n FROM {CAT}.obs_incidents i
                  WHERE i.detector = 'latency_anomaly'
                    AND i.severity = 'CRITICAL'
                    AND NOT exists (SELECT 1 FROM {CAT}.latency_anomalies la
                                    WHERE cast(la.id AS STRING) = i.source_row_id
                                      AND la.latency_verdict = 'anomaly')""")[0]["n"]
    record("P2 latency incidents match the alert's severity logic", n_bad == 0,
           f"{n_bad} CRITICAL latency incident(s) whose verdict is not 'anomaly' — add "
           f"WHERE latency_verdict = 'anomaly' to that INCIDENT_SOURCES entry", level="WARN")
except Exception as e:
    record("P2 latency incident severity", False, str(e)[:150])

# P2: the watermark should exist and be recent -- a stale watermark left at a backfill
# reset would mean a detector is silently reprocessing or skipping data it shouldn't.
try:
    rows = q(f"SELECT detector, last_processed_ts, updated_at FROM {CAT}._obs_watermark")
    record("P2 watermark exists", len(rows) > 0,
           "; ".join(f"{r['detector']} @ {r['last_processed_ts']}" for r in rows))
    stale = [r for r in rows
             if r["updated_at"] and (r["last_processed_ts"] is None
                                     or (r["updated_at"] - r["last_processed_ts"]).total_seconds() > 7200)]
    record("P2 watermark is current (not left at a backfill reset)", not stale,
           "; ".join(f"{r['detector']} lagging" for r in stale) or "current", level="WARN")
except Exception as e:
    record("P2 _obs_watermark", False, f"MISSING: {str(e)[:150]}")

# COMMAND ----------

# MAGIC %md ## Phase 3 -- alert notebook structural completeness

# COMMAND ----------

# P3: incident-source completeness (Phase 6 item 2 of the approved plan). Every
# INCIDENT_SOURCES entry with severity='CRITICAL' is what actually reaches Teams (see
# ptof_obs_alert.ipynb's notify cell, which filters WHERE severity = 'CRITICAL') -- so it needs
# both a DETECTOR_META card (a real label/what, not the bare detector name) and a BACKTRACK
# query (a "where to look" pointer), or a real finding renders a degraded Teams card.
#
# Static source check, not a live execution: reads ptof_obs_alert.ipynb's own cell source via
# ast, parsing out INCIDENT_SOURCES/DETECTOR_META/BACKTRACK's literal structure without ever
# calling spark.sql(...)/requests.post(...) -- this cannot accidentally trigger the alert
# notebook's MERGE or Teams POST side effects.
import ast
import json
import re

ALERT_NB_PATH = ("/Workspace/Users/tyler.kei@lilly.com/ptof_agent_observability_repo/"
                 "ptof_obs_alert.ipynb")


def _alert_cell_sources(nb_path):
    with open(nb_path) as f:
        alert_nb = json.load(f)
    return ["".join(c["source"]) if isinstance(c["source"], list) else c["source"]
            for c in alert_nb["cells"] if c.get("cell_type") == "code"]


def _assigned_literal(src, name):
    """Locate `name = <literal>` in source via AST, without executing anything."""
    tree = ast.parse(src)
    for node in ast.walk(tree):
        if (isinstance(node, ast.Assign) and len(node.targets) == 1
                and isinstance(node.targets[0], ast.Name) and node.targets[0].id == name):
            return node.value
    return None


def _dict_keys(dict_node):
    return {n.value for n in dict_node.keys if isinstance(n, ast.Constant)}


def _incident_sources_critical(list_node):
    out = set()
    for tup in list_node.elts:
        if isinstance(tup, ast.Tuple):
            detector, severity = tup.elts[0].value, tup.elts[4].value
            if severity == "CRITICAL":
                out.add(detector)
    return out


try:
    sources = _alert_cell_sources(ALERT_NB_PATH)
    incident_node = meta_node = backtrack_node = None
    for s in sources:
        incident_node = incident_node or _assigned_literal(s, "INCIDENT_SOURCES")
        backtrack_node = backtrack_node or _assigned_literal(s, "BACKTRACK")
        meta_node = meta_node or _assigned_literal(s, "DETECTOR_META")
    assert incident_node is not None, "INCIDENT_SOURCES not found in ptof_obs_alert.ipynb"
    assert backtrack_node is not None, "BACKTRACK not found in ptof_obs_alert.ipynb"
    assert meta_node is not None, "DETECTOR_META not found in ptof_obs_alert.ipynb"

    critical = _incident_sources_critical(incident_node)
    meta_keys = _dict_keys(meta_node)
    backtrack_keys = _dict_keys(backtrack_node)

    missing_meta = critical - meta_keys
    missing_backtrack = critical - backtrack_keys
    record("P3 every CRITICAL INCIDENT_SOURCES entry has a DETECTOR_META card",
           not missing_meta,
           f"missing={sorted(missing_meta)}" if missing_meta else f"{len(critical)} covered")
    record("P3 every CRITICAL INCIDENT_SOURCES entry has a BACKTRACK query",
           not missing_backtrack,
           f"missing={sorted(missing_backtrack)}" if missing_backtrack else f"{len(critical)} covered")
except Exception as e:
    record("P3 incident-source completeness (static check)", False, str(e)[:200])

# P3: alerting_pipeline synthetic-failure detector (Phase 6 item 2) -- verifies post_teams's
# final-failure MERGE (source_row_id = 'teams_notify_failed_<day>') dedupes on its per-day key
# rather than duplicating, and that any row present has the shape the MERGE always writes.
# 0 rows is the expected/current state -- WARN, not FAIL, until a real webhook failure occurs
# or the synthetic Teams test (Phase 6 step 3, run separately with confirmation) exercises it.
try:
    rows = q(f"""SELECT source_row_id, detection_count, severity, capability
                 FROM {CAT}.obs_incidents WHERE detector = 'alerting_pipeline'""")
    n = len(rows)
    dup_keys = n - len({r["source_row_id"] for r in rows})
    bad_shape = [r for r in rows if r["severity"] != "CRITICAL" or r["capability"] is not None
                 or not re.match(r"^teams_notify_failed_\d{4}-\d{2}-\d{2}$",
                                 r["source_row_id"] or "")]
    record("P3 alerting_pipeline has no duplicate per-day rows (MERGE dedup)", dup_keys == 0,
           f"{dup_keys} duplicate source_row_id(s) among {n} row(s)")
    record("P3 alerting_pipeline row shape matches post_teams's MERGE",
           len(bad_shape) == 0,
           f"{len(bad_shape)} malformed row(s)" if bad_shape else
           (f"{n} row(s), none yet -- expected until a real/synthetic Teams failure occurs"
            if n == 0 else f"{n} row(s) well-formed"), level="WARN")
except Exception as e:
    record("P3 alerting_pipeline synthetic-failure detector", False, str(e)[:200])

# COMMAND ----------

# MAGIC %md ## Hygiene

# COMMAND ----------

names = {r["tableName"] for r in q(f"SHOW TABLES IN {CAT}")}
# Hygiene: tables that should have been renamed/dropped during this engagement should
# stay gone, and tables this design deliberately kept for future use should stay present.
record("stale transport_allowlist absent", "transport_allowlist" not in names,
       "BACK AGAIN — the seed cell is still in task 03" if "transport_allowlist" in names else "gone")
record("orphaned success_rate_daily absent", "success_rate_daily" not in names,
       "still present" if "success_rate_daily" in names else "gone", level="WARN")
record("P1.8 ish_entity_dim retained", table_exists("ish_entity_dim"),
       f"rows={count_of('ish_entity_dim')}")
record("P1.9 handover_delivery_failures exists", table_exists("handover_delivery_failures"),
       f"rows={count_of('handover_delivery_failures')} — real, unresolved SMTP failures")

# COMMAND ----------

# MAGIC %md ## Summary

# COMMAND ----------

order = {"FAIL": 0, "WARN": 1, "PASS": 2}
checks.sort(key=lambda c: order[c["status"]])
print(f"{'status':<7} {'item':<62} detail")
print("-" * 145)
for c in checks:
    print(f"{c['status']:<7} {c['item']:<62} {c['detail']}")

fails = [c for c in checks if c["status"] == "FAIL"]
warns = [c for c in checks if c["status"] == "WARN"]
print("\n" + "=" * 145)
print(f"{len(checks)-len(fails)-len(warns)} PASS · {len(warns)} WARN · {len(fails)} FAIL")
if fails:
    print("\nBLOCKING:")
    for c in fails:
        print(f"  - {c['item']}: {c['detail']}")

# COMMAND ----------

# MAGIC %md ## Not checkable from data — verify in the Jobs UI
# MAGIC
# MAGIC | Item | Where | Expected |
# MAGIC |---|---|---|
# MAGIC | 2.1 consolidated compute | latest run → Compute | **one** cluster, duration ~4–5 min |
# MAGIC | 2.5 trigger throttling | Schedules & Triggers → Advanced | wait 30 s, min between 300 s |
# MAGIC | 0.1b task 02 path | Tasks → `02_latency_detection` | ends `agent_obs_latency_detection` |
# MAGIC | 0.3 notifications | Job notifications | a destination on **Failure** |
# MAGIC | 0.5c job parameters | any task run → Parameters | `env`, `lookback_minutes`, `alert_webhook` resolved |
# MAGIC | `obs_setup_seed` | Workspace | exists and is **not** a job task |
# MAGIC | Task 06 | latest run | **green** — raises only on UNAVAILABLE since Phase 2 |
# MAGIC
# MAGIC ## Expected WARNs — not defects
# MAGIC - `capability_health` flagging `dsa_optimize` on `test-bot-claude-v1`: 1 success in 22 calls.
# MAGIC - `handover_delivery_failures` non-zero: 15 handovers never reached
# MAGIC   `PFS3_ISH_SME@lists.lilly.com`, ~9.3% of all send attempts since Jun 19.
# MAGIC - `shift_context_missing` non-zero: all `dsa_*` rows write NULL/'' for shift_date,
# MAGIC   shift_type and batch_nbr, which is why `rapid_human_correction` is parked.